# Axon cloud-worker preflight

This notebook checks the **existing governed English D64 trainer** on a fresh Colab worker and saves its evidence to Google Drive. It runs **zero optimizer steps**, does not resume the failed step-24 experiment, and does not produce a mobile Core. The phone is the intended runtime host; this notebook is a temporary worker check.

Run the cells in order. Google will ask you to authorize your own Drive. No PC or Cloudflare is required. A successful preflight is not a training/mastery pass.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess, sys, uuid, json, hashlib, shutil
SOURCE_COMMIT = '843bbedd25347d367da1f979488a6ecb77128ab7'
repo = Path('/content/axon-phone-worker')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Axepapag/Axon.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
import torch
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
job_id = 'preflight-' + str(uuid.uuid4())
job = Path('/content') / job_id
job.mkdir()
state = job / 'State'
report = job / 'report.json'


In [ ]:
command = [sys.executable, str(repo / 'scripts/train_living_reasoning_smoke.py'),
           '--state-root', str(state), '--device', 'cpu', '--curriculum', 'substrate',
           '--preflight-only', '--report', str(report)]
result = subprocess.run(command, cwd=repo, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(job / 'worker.log').write_text(result.stdout)
print(result.stdout)
print('Exit code:', result.returncode)
# Preserve failure evidence too; the next cell never labels failure as acceptance.


In [ ]:
destination = Path('/content/drive/MyDrive/Axon/Training/preflight') / job_id
destination.mkdir(parents=True, exist_ok=False)
archive = Path(shutil.make_archive(str(job), 'zip', job))
payload = destination / 'evidence.zip'
shutil.copyfile(archive, payload)
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
assert digest(payload) == digest(archive)
manifest = {'schema': 'axon-cloud-preflight-evidence-v1', 'job_id': job_id,
            'source_commit': SOURCE_COMMIT, 'exit_code': result.returncode,
            'optimizer_steps': 0, 'mobile_core_produced': False,
            'artifact': 'evidence.zip', 'sha256': digest(payload), 'complete': True}
# Completion manifest last; it attests file completion, not successful training.
(destination / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Evidence saved:', destination)
if result.returncode:
    raise RuntimeError('Preflight failed; evidence saved. No training or promotion occurred.')
